# Best variant on the held-out test videos (Kaggle, self-contained)

**OSTrack + V-JEPA re-localization verified by OSTrack (score >= 0.6)** - the best variant on the 40 dev
videos (AUC 57.61 vs 57.09). Its threshold was chosen after looking at dev, so here it runs unchanged on
the **240 other LaSOT test videos** (`configs/splits/test_holdout.txt`) to check that the gain is real.
The baseline on these videos is already computed (`results/ostrack_ours`).

The 240 videos are split into `N_PARTS = 2` parts of ~3.5 h each: make one version with `PART = 0` and one
with `PART = 1` (in parallel if Kaggle allows two GPU sessions). Videos are processed in chunks of 20
(V-JEPA cache -> tracking -> cache deleted), and results are zipped after every chunk into
`/kaggle/working/holdout_part<PART>.zip`.

Settings: **Accelerator = GPU T4 x2**, **Internet = On**, no inputs. Then **Save Version -> Save & Run All**.

In [ ]:
PART, N_PARTS = 0, 2   # <- set PART = 1 for the second version
CHUNK = 20

In [ ]:
import socket
try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError:
    raise SystemExit('NO INTERNET: Settings (right panel) -> Internet = On, then run again.')
import torch
if torch.cuda.device_count() == 0:
    raise SystemExit('NO GPU: Settings (right panel) -> Accelerator = GPU T4 x2, then run again.')
print(torch.cuda.device_count(), 'GPUs:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/MariiaHamanuk/Object-Tracking.git',
                '/kaggle/working/Object-Tracking'], check=True)
!pip install -q timm==0.6.13 easydict visdom tensorboardX jpeg4py lmdb gdown "transformers>=4.53"
!gdown 1dySfPrSk-knAEo0lTXqZ1kMk3BCvVofV -O /kaggle/working/OSTrack_ep0300.pth.tar

In [ ]:
import os, glob, shutil, numpy as np
os.chdir('/kaggle/working/Object-Tracking')
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.makedirs('/kaggle/tmp', exist_ok=True)
W = '/kaggle/working'
CKPT = f'{W}/OSTrack_ep0300.pth.tar'
CACHE = '/kaggle/tmp/cache/vjepa'
OUT = f'{W}/results_raw/ostrack_vjepa_reloc_verify_min/lasot'
BEST = ['--reloc-verify', '3', '--verify-min', '0.6']   # the best dev variant, unchanged
N_GPU = max(torch.cuda.device_count(), 1)
os.makedirs(f'{W}/logs', exist_ok=True)

def run_shards(name, args):
    procs = []
    for i in range(N_GPU):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
        log = open(f'{W}/logs/{name}_shard{i}.log', 'a')
        procs.append(subprocess.Popen(['python'] + args + ['--shard', str(i), '--num-shards', str(N_GPU)],
                                      env=env, stdout=log, stderr=subprocess.STDOUT))
    codes = [p.wait() for p in procs]
    os.system(f"grep -hE 'FAILED|Error|Traceback' {W}/logs/{name}_shard*.log | tail -n 5")
    print(name, 'exit codes', codes)
    return codes

def pack():
    os.system(f'cd {W} && rm -f holdout_part{PART}.zip && zip -qr holdout_part{PART}.zip results_raw logs; '
              f'ls -la {W}/holdout_part{PART}.zip')

seqs = [s.strip() for s in open('configs/splits/test_holdout.txt') if s.strip()][PART::N_PARTS]
print(f'part {PART}/{N_PARTS}: {len(seqs)} videos')

## Self-test (30 frames, random V-JEPA maps): the best variant runs end to end

In [ ]:
os.makedirs('/kaggle/tmp/selftest/cache', exist_ok=True)
np.savez('/kaggle/tmp/selftest/cache/airplane-1.npz', frames=np.arange(0, 3000, 8),
         maps=np.random.randn(375, 16, 16, 1024).astype(np.float16))
subprocess.run(['python', 'scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', 'airplane-1',
                '--max-frames', '30', '--vjepa-cache', '/kaggle/tmp/selftest/cache', '--out', '/kaggle/tmp/selftest/best',
                '--work', '/kaggle/tmp/selftest/work'] + BEST, check=True)
assert np.loadtxt('/kaggle/tmp/selftest/best/airplane-1.txt', delimiter='\t').shape == (30, 4)
print('SELF-TEST OK')

## Run: per chunk of 20 videos, V-JEPA cache -> tracking -> delete the cache

In [ ]:
for c in range(0, len(seqs), CHUNK):
    chunk = [s for s in seqs[c:c + CHUNK] if not os.path.exists(f'{OUT}/{s}.txt')]
    if not chunk:
        continue
    split = f'/kaggle/tmp/chunk.txt'
    open(split, 'w').write('\n'.join(chunk) + '\n')
    print(f'chunk {c // CHUNK + 1}/{-(-len(seqs) // CHUNK)}: {len(chunk)} videos', flush=True)
    for attempt in range(2):  # a re-run only does the videos that failed to download
        run_shards('vjepa_cache', ['scripts/cache_vjepa.py', '--sequences', split, '--out', CACHE, '--work', '/kaggle/tmp'])
        if all(os.path.exists(f'{CACHE}/{s}.npz') for s in chunk):
            break
    run_shards('best', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', split,
                        '--vjepa-cache', CACHE, '--out', OUT, '--work', '/kaggle/tmp'] + BEST)
    shutil.rmtree(CACHE, ignore_errors=True)
    pack()
done = [s for s in seqs if os.path.exists(f'{OUT}/{s}.txt')]
print(f'{len(done)} of {len(seqs)} videos done')
os.system(f'rm -rf {W}/Object-Tracking {CKPT}')